Working with Text Data 
Les model ne compreennent pas les mots naturellment, on utilise un concept appeller token = sous mots ou mot, l'idée est de vectoriser ce token pour capturer 
son sens/signification,  



In [5]:


# [Texte Brut]
#    "journey"
#        ↓
#        ↓ (Le Tokenizer traduit le mot en ID numérique)
#        ↓
# [ID unique]
#    4920 (l'identifiant du mot "journey" dans GPT-2)
#        ↓
#        ↓ (Le Dataset le place dans un lot à la position d'index 1)
#        ↓
# [Position dans le Lot]
#    Index de token : 4920  |  Index de position : 1
#        ↓
#        ↓ (Passage dans les deux tables d'Embedding en parallèle)
#        ↓
# [Calcul des Embeddings (vecteurs de taille 256)]
#    • Token Embedding (ligne 4920) ──> [ 0.91,  1.58, -0.45, ... ] (Sens de "journey")
#                                                +
#    • Position Embedding (ligne 1) ──> [ -0.12, 0.34,  0.88, ... ] (Position numéro 2)
#        ↓
#        ↓ (Addition des deux vecteurs élément par élément)
#        ↓
# [Vecteur final (taille 256)]
#    [ 0.79,  1.92,  0.43, ... ]
#        ↓
#        ↓ (Ce vecteur est rangé dans le tenseur 3D final à l'emplacement [Batch_i, 1, :])
#        ↓
# [Tenseur 3D prêt pour le LLM]
#    Shape: [Batch_size, Contexte, 256]


import os
import re
import requests
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader

# =====================================================================
# SECTION 1 : TÉLÉCHARGEMENT ET CHARGEMENT DU TEXTE BRUT
# =====================================================================
def get_raw_text():
    file_path = "the-verdict.txt"
    if not os.path.exists(file_path):
        print("[INFO] Téléchargement de 'the-verdict.txt'...")
        url = (
            "https://raw.githubusercontent.com/rasbt/"
            "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
            "the-verdict.txt"
        )
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        with open(file_path, "wb") as f:
            f.write(response.content)
        print("[INFO] Téléchargement réussi !")
        
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
    return text


# =====================================================================
# SECTION 2 : TOKENIZER DE ZÉRO (FROM SCRATCH)
# =====================================================================

# Version 1 : Tokenizer basique (va échouer si un mot est inconnu)
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}
    
    def encode(self, text):
        # Découpe le texte par mots, espaces et ponctuation
        preprocessed = re.split(r'([,.:;?_!"()\'\]|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        # Conversion directe (crachera en cas d'erreur de clé si mot absent)
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Nettoie les espaces superflus avant la ponctuation
        text = re.sub(r'\s+([,.:;?!"()\'\]])', r'\1', text)
        return text


# Version 2 : Tokenizer robuste gérant les jetons spéciaux (<|unk|> et <|endoftext|>)
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\'\]|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        
        # Gestion des mots inconnus : s'ils ne sont pas dans notre dictionnaire
        # on les remplace par le jeton spécial "<|unk|>"
        preprocessed = [
            item if item in self.str_to_int else "<|unk|>" 
            for item in preprocessed
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'\]])', r'\1', text)
        return text


# Fonction pour extraire et construire le dictionnaire pour le Tokenizer de zéro
def build_custom_vocab(raw_text):
    preprocessed = re.split(r'([,.:;?_!"()\'\]|--|\s)', raw_text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    all_tokens = sorted(list(set(preprocessed)))
    
    # Extension du vocabulaire avec les jetons spéciaux
    all_tokens.extend(["<|endoftext|>", "<|unk|>"])
    vocab = {token: idx for idx, token in enumerate(all_tokens)}
    return vocab


# =====================================================================
# SECTION 3 : PRÉPARATION DU DATASET ET DU DATALOADER (FENÊTRE GLISSANTE)
# =====================================================================

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenisation avec le tokenizer passé en argument
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        
        assert len(token_ids) > max_length, (
            "Le texte est trop court pour la longueur de contexte choisie."
        )

        # Découpe le texte en blocs glissants
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1] # Décalé de +1 vers la droite
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    
    # Utilisation du BPE (BytePair Encoding) de GPT-2
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader


# =====================================================================
# SECTION 4 : POINT D'ENTRÉE ET EXÉCUTION DU PIPELINE COMPLET
# =====================================================================
if __name__ == "__main__":
    
    # --- 1. Chargement des données brutes ---
    raw_text = get_raw_text()
    print(f"[1] Fichier chargé : {len(raw_text)} caractères.")

    # --- 2. Démonstration des Tokenizers Custom ---
    print("\n[2] --- DÉMO DES TOKENIZERS CUSTOM ---")
    custom_vocab = build_custom_vocab(raw_text)
    
    # Test V1 (va échouer sur "Hello" car il n'est pas dans the-verdict.txt)
    tokenizer_v1 = SimpleTokenizerV1(custom_vocab)
    try:
        tokenizer_v1.encode("Hello, how are you?")
    except KeyError as e:
        print(f"-> V1 a échoué comme prévu : KeyError sur le mot {e}")
        
    # Test V2 (gère "Hello" en le remplaçant par <|unk|>)
    tokenizer_v2 = SimpleTokenizerV2(custom_vocab)
    sample_text = "Hello, do you like tea? <|endoftext|> In the garden."
    encoded_v2 = tokenizer_v2.encode(sample_text)
    decoded_v2 = tokenizer_v2.decode(encoded_v2)
    print(f"-> Texte d'origine : '{sample_text}'")
    print(f"-> Encodé avec V2 : {encoded_v2}")
    print(f"-> Décodé avec V2 : '{decoded_v2}'")

    # --- 3. Démonstration du BytePair Encoding (BPE avec tiktoken) ---
    print("\n[3] --- DÉMO BPE (TIKTOKEN) ---")
    bpe_tokenizer = tiktoken.get_encoding("gpt2")
    bpe_text = "Hello, do you like tea? <|endoftext|> In someunknownPlace."
    bpe_encoded = bpe_tokenizer.encode(bpe_text, allowed_special={"<|endoftext|>"})
    bpe_decoded = bpe_tokenizer.decode(bpe_encoded)
    print(f"-> Encodé avec BPE : {bpe_encoded}")
    print(f"-> Décodé avec BPE : '{bpe_decoded}'")

    # --- 4. Démonstration du DataLoader avec Décalage (Shift) ---
    print("\n[4] --- DÉMO DATALOADER (Next-Token Prediction) ---")
    # Configuration : lots de 8, contexte de 4 tokens, et stride de 4
    # Note : stride = max_length évite le chevauchement pour limiter l'overfitting
    dataloader = create_dataloader_v1(
        raw_text, batch_size=8, max_length=4, stride=4, shuffle=False
    )
    
    data_iter = iter(dataloader)
    x_batch, y_batch = next(data_iter)
    
    print("Forme du lot d'entrées (x) :", x_batch.shape)  # [8, 4]
    print("Forme du lot de cibles (y) :", y_batch.shape)  # [8, 4]
    print("\nExemple de lot d'entrée x_batch :\n", x_batch)
    print("\nExemple de lot cible y_batch :\n", y_batch)

    # --- 5. Couches d'Embedding (Token + Position) ---
    print("\n[5] --- DÉMO COUCHES EMBEDDING (GPT-2) ---")
    
    VOCAB_SIZE = 50257  # Taille du dictionnaire GPT-2
    OUTPUT_DIM = 256    # Taille du vecteur représentant chaque mot
    CONTEXT_LENGTH = 4  # Taille du contexte
    
    # A. Initialisation des couches d'embedding
    torch.manual_seed(123)
    token_embedding_layer = torch.nn.Embedding(VOCAB_SIZE, OUTPUT_DIM)
    pos_embedding_layer = torch.nn.Embedding(CONTEXT_LENGTH, OUTPUT_DIM)
    
    # B. Passage des tokens à la couche d'embedding
    # x_batch est de taille [8, 4]. Chaque ID va récupérer sa ligne de taille 256.
    token_embeddings = token_embedding_layer(x_batch)
    print("Forme des Token Embeddings :", token_embeddings.shape) # [8, 4, 256]
    
    # C. Création des vecteurs de position
    # On crée une liste d'indices de positions : [0, 1, 2, 3]
    position_ids = torch.arange(CONTEXT_LENGTH)
    pos_embeddings = pos_embedding_layer(position_ids)
    print("Forme des Positional Embeddings :", pos_embeddings.shape) # [4, 256]
    
    # D. Fusion par Addition
    # Grâce au broadcasting de PyTorch, le tenseur de position [4, 256] 
    # est automatiquement dupliqué et additionné à chaque ligne du batch [8, 4, 256]
    final_input_embeddings = token_embeddings + pos_embeddings
    
    print("Forme finale des Embeddings en entrée du LLM :", final_input_embeddings.shape) # [8, 4, 256]
    print("\n[SUCCÈS] Le pipeline d'entrée du LLM est entièrement fonctionnel !")


[INFO] Téléchargement de 'the-verdict.txt'...
[INFO] Téléchargement réussi !
[1] Fichier chargé : 20479 caractères.

[2] --- DÉMO DES TOKENIZERS CUSTOM ---


/tmp/ipykernel_4955/1581672164.py:114: FutureWarning: Possible set difference at position 17
  preprocessed = re.split(r'([,.:;?_!"()\'\]|--|\s)', raw_text)


error: bad character range |-- at position 16

mechanism d'attention

In [ ]:
import math
import torch
import torch.nn as nn

# =====================================================================
# SECTION 1 : ATTENTION SIMPLE SANS POIDS ENTRAÎNABLES (Section 3.3)
# =====================================================================
def simple_self_attention_demo():
    # Phrase d'exemple : "Your journey starts with one step" (6 tokens)
    # Chaque token est représenté par un vecteur d'embedding de taille 3
    inputs = torch.tensor(
        [[0.43, 0.15, 0.89],  # Your
         [0.55, 0.87, 0.66],  # journey
         [0.57, 0.85, 0.64],  # starts
         [0.22, 0.58, 0.33],  # with
         [0.77, 0.25, 0.10],  # one
         [0.05, 0.80, 0.55]]  # step
    )
    
    # Étape 1 : Calcul des scores d'attention non-normalisés (produit matriciel X * X^T)
    attn_scores = inputs @ inputs.T
    
    # Étape 2 : Normalisation avec Softmax le long des lignes (les poids somment à 1)
    attn_weights = torch.softmax(attn_scores, dim=-1)
    
    # Étape 3 : Calcul des vecteurs de contexte par moyenne pondérée des entrées
    context_vecs = attn_weights @ inputs
    
    return inputs, context_vecs


# =====================================================================
# SECTION 2 : ATTENTION SIMPLE AVEC POIDS ENTRAÎNABLES (Section 3.4)
# =====================================================================

# Version utilisant nn.Linear pour plus de stabilité et d'initialisation propre
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        # Produit scalaire Scalé (Scaled Dot-Product)
        attn_scores = queries @ keys.transpose(-2, -1)
        # Division par racine(d_k) pour éviter l'explosion des gradients
        d_k = keys.shape[-1]
        attn_weights = torch.softmax(attn_scores / math.sqrt(d_k), dim=-1)

        context_vec = attn_weights @ values
        return context_vec


# =====================================================================
# SECTION 3 : ATTENTION CAUSALE ET DROPOUT (Section 3.5)
# =====================================================================

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
        # Dropout pour réduire l'overfitting lors de l'entraînement
        self.dropout = nn.Dropout(dropout)
        
        # Le masque causal : matrice triangulaire supérieure de zéros et uns
        # On l'enregistre en tant que buffer (ne sera pas mis à jour par rétropropagation)
        self.register_buffer(
            'mask', 
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Calcul des scores : transposition le long des dimensions de jetons
        attn_scores = queries @ keys.transpose(1, 2)
        
        # Masquage des mots futurs : remplace les 1 du masque par -infini
        # Ainsi, e^(-inf) = 0 après softmax.
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -float('inf'))
        
        # Normalisation Softmax + Dropout
        attn_weights = torch.softmax(attn_scores / math.sqrt(keys.shape[-1]), dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec


# =====================================================================
# SECTION 4 : MULTI-HEAD ATTENTION (Section 3.6)
# =====================================================================

# Option A : Wrapper naïf en empilant plusieurs modules CausalAttention
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        # On crée une liste de modules d'attention indépendants
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) 
             for _ in range(num_heads)]
        )

    def forward(self, x):
        # Concaténation des résultats de chaque tête sur la dimension finale (embedding)
        return torch.cat([head(x) for head in self.heads], dim=-1)


# Option B (Standard dans les LLM) : Stand-alone MultiHeadAttention avec division des poids
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out doit être divisible par num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        # Dimension projetée de chaque tête
        self.head_dim = d_out // num_heads 

        # Projections linéaires uniques de taille d_out (englobant toutes les têtes)
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
        # Projection finale de sortie pour fusionner l'information des têtes
        self.out_proj = nn.Linear(d_out, d_out)  
        self.dropout = nn.Dropout(dropout)
        
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        # 1. Calcul des projections globales
        # Shape: (b, num_tokens, d_out)
        keys = self.W_key(x) 
        queries = self.W_query(x)
        values = self.W_value(x)

        # 2. Division logique en têtes en changeant la vue
        # (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # 3. Transposition pour placer la dimension num_heads en position 1
        # (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # 4. Produit scalaire d'attention par tête (Q @ K^T)
        # Shape de k.transpose(2, 3) : (b, num_heads, head_dim, num_tokens)
        attn_scores = queries @ keys.transpose(2, 3) # Shape: (b, num_heads, num_tokens, num_tokens)

        # 5. Application du masque causal
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -float('inf'))
        
        # 6. Softmax + Dropout
        attn_weights = torch.softmax(attn_scores / math.sqrt(self.head_dim), dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 7. Pondération avec les valeurs et remise en forme
        # context_vec shape: (b, num_heads, num_tokens, head_dim) -> (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Concaténation logique des têtes : (b, num_tokens, d_out)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        
        # 8. Projection linéaire finale de sortie
        return self.out_proj(context_vec)


# =====================================================================
# CODE DE TEST ET DÉMONSTRATION
# =====================================================================
if __name__ == "__main__":
    
    # --- TEST SECTION 1 ---
    inputs, simple_out = simple_self_attention_demo()
    print("Inputs shape :", inputs.shape)
    print("Simple Self-Attention Context Vectors shape :", simple_out.shape)
    
    # --- PRÉPARATION D'UN BATCH SIMULÉ POUR LES MODÈLES SUIVANTS ---
    # Nous simulons un batch de 2 exemples contenant les 6 tokens d'origine
    batch = torch.stack((inputs, inputs), dim=0) # Shape: [2, 6, 3]
    print("\nBatch Input shape pour l'attention causale :", batch.shape)
    
    # --- TEST SECTION 3 : CAUSAL ATTENTION ---
    print("\n--- TEST CAUSAL ATTENTION (Single-Head) ---")
    torch.manual_seed(123)
    causal_attn = CausalAttention(d_in=3, d_out=2, context_length=6, dropout=0.0)
    causal_out = causal_attn(batch)
    print("Sortie CausalAttention shape :", causal_out.shape) # Shape: [2, 6, 2]
    
    # --- TEST SECTION 4 : MULTI-HEAD ATTENTION (WRAPPER) ---
    print("\n--- TEST MULTI-HEAD ATTENTION (WRAPPER) ---")
    torch.manual_seed(123)
    mha_wrapper = MultiHeadAttentionWrapper(d_in=3, d_out=2, context_length=6, dropout=0.0, num_heads=2)
    wrapper_out = mha_wrapper(batch)
    print("Sortie Wrapper (concaténation de 2 têtes de dim 2) shape :", wrapper_out.shape) # Shape: [2, 6, 4]
    
    # --- TEST SECTION 4 : MULTI-HEAD ATTENTION (STANDARD) ---
    print("\n--- TEST MULTI-HEAD ATTENTION (STANDARD AVEC SPLIT DE POIDS) ---")
    torch.manual_seed(123)
    # Ici, d_out=4 est divisé entre 2 têtes (head_dim = 2)
    mha_standard = MultiHeadAttention(d_in=3, d_out=4, context_length=6, dropout=0.0, num_heads=2)
    standard_out = mha_standard(batch)
    print("Sortie MultiHeadAttention standard (dim globale 4) shape :", standard_out.shape) # Shape: [2, 6, 4]
    print("\n[SUCCÈS] Tous les mécanismes d'attention s'exécutent correctement !")


exercice codé un self attention causal

In [ ]:
import torch 
import torch.nn as nn 
import math  # Nécessaire pour math.sqrt
import torch.functional as F

class CausalAttention(nn.Module):  # Majuscule (convention Python)
    def __init__(self, d_in, d_out, dropout, context_length, qkv_bias=False): 
        super().__init__()
        self.d_out = d_out
        
        # Définition des projections linéaires
        self.wq = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.wk = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.wv = nn.Linear(d_in, d_out, bias=qkv_bias)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
        # Enregistrement du masque causal
        self.register_buffer(
            'mask', 
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
        
    def forward(self, x): 
        # x a pour dimension (batch_size, num_tokens, d_in)
        b, num_tokens, d_in = x.shape 
        
        Q = self.wq(x)  # (b, num_tokens, d_out)
        K = self.wk(x)  # (b, num_tokens, d_out)
        V = self.wv(x)  # (b, num_tokens, d_out)

        # Produit scalaire Q @ K^T. 
        # On transpose les dimensions 1 (num_tokens) et 2 (d_out) de K.
        attn_scores = Q @ K.transpose(1, 2)  # (b, num_tokens, num_tokens)
        
        # Masquage des mots futurs
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -float('inf'))
        
        # Normalisation Softmax (divisé par la racine de la dimension des clés d_out)
        attn_weights = torch.softmax(attn_scores / math.sqrt(self.d_out), dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Calcul du vecteur de contexte
        context_vec = attn_weights @ V  # (b, num_tokens, d_out)
        return context_vec


class MLP(nn.Module): 
    def __init__(self,d_in, hidden1, hidden2, d_out, dropout = 0.2):
        super().__init_()

        self.l1 = nn.Linear(d_in, hidden1)
        self.l2 = nn.Linear(hidden1, hidden2)
        self.out = nn.Linear(hidden2, d_out)
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        dropout = self.dropout
        x = dropout(self.l2(F.relu(self.l1(x)))) 
        x = self.out(x)
        return x 


class BlockTransformer(nn.Module):
    def __init__(self,d_model, context_length, dropout=0.2):

        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        self.attention = CausalAttention(
        d_in=d_in, 
        d_out=d_out, 
        dropout=dropout_rate, 
        context_length=context_length
)
        self.mlp = MLP(
            d_in=d_model, 
            hidden1=d_model * 4, 
            hidden2=d_model * 4, 
            d_out=d_model, 
            dropout=dropout
        )
    def forward(self,x): 
 
        x = x + self.attention(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x 
# -----------------
# TEST DU CODE
# -----------------

# 1. Définition des paramètres
batch_size = 2
context_length = 2048
d_in = 512
d_out = 512
dropout_rate = 0.4

# 2. Création de l'entrée tensor de dimension (batch, sequence_length, d_in)
# Ici: batch=2, sequence_length=6 tokens, dimension=512
x = torch.randn(batch_size, 6, d_in)
print("1. Entrée x :", x.shape)

# 3. Initialisation du modèle avec ses arguments


# 4. Appel du forward pass
model = BlockTransformer(d_model=d_in, context_length=context_length, dropout=dropout_rate)
predict = model(x)
print("2. Sortie (context_vec) :", predict.shape)


1. Entrée x : torch.Size([2, 6, 512])
2. Sortie (context_vec) : torch.Size([2, 6, 512])
